
# Géographie, distances et temps d'accès

Ce notebook construit la composante géographique de l'indicateur d'accessibilité aux soins.

## Objectif

Pour chaque commune d'origine, identifier les communes accessibles en voiture en **20 minutes maximum** et produire :

```text
CODGEO_origine
CODGEO_destination
distance_vol_oiseau_km
distance_route_km
temps_route_min
classe_temps
poids_distance
```

Les temps routiers sont calculés avec le serveur **Metric-OSRM de l'Insee déployé sur SSPCloud**.

La pondération retenue pour les médecins généralistes est :

- jusqu'à 10 minutes : poids `1`
- de 10 à 15 minutes : poids `2/3`
- de 15 à 20 minutes : poids `1/3`
- au-delà de 20 minutes : poids `0`

Le notebook ne conserve donc dans la table finale que les couples de communes accessibles en 20 minutes maximum.

## Principe de calcul

Pour éviter de demander à OSRM tous les couples de communes de France :

1. on récupère un point de référence pour chaque commune ;
2. on effectue un préfiltrage à vol d'oiseau dans un rayon de 50 km ;
3. on calcule ensuite le vrai trajet routier uniquement pour ces communes candidates ;
4. on conserve les trajets dont le temps routier est inférieur ou égal à 20 minutes.

Le rayon de 50 km est uniquement un filtre informatique : le critère final reste toujours le **temps routier OSRM**.

## Attention
Le calcul complet des temps routiers est coûteux. Le fichier final généré est sauvegardé dans processed/ et peut être réutilisé directement par les notebooks suivants sans recalcul.


In [2]:
import os
import json
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from sklearn.neighbors import BallTree
from tqdm.auto import tqdm


## 2. Paramètres

In [ ]:
URL_INSEE = (
    "https://minio.lab.sspcloud.fr/mangane/"
    "Projet_PDS_2026/processed/insee_communes_final.csv"
)

# API officielle de découpage administratif 
URL_COMMUNES = "https://geo.api.gouv.fr/communes"

# Serveur Metric-OSRM de l'Insee sur SSPCloud
OSRM_SERVER = "https://metric-osrm-backend.lab.sspcloud.fr"
OSRM_PROFILE = "driving"


RAYON_PREFILTRE_KM = 50


MAX_DESTINATIONS_PAR_REQUETE = 60

MAX_RETRIES = 5
PAUSE_ENTRE_REQUETES = 0.10

DOSSIER_LOCAL = Path("data/processed/distances")
DOSSIER_BATCHES = DOSSIER_LOCAL / "batches"
DOSSIER_LOCAL.mkdir(parents=True, exist_ok=True)
DOSSIER_BATCHES.mkdir(parents=True, exist_ok=True)


FICHIER_COORDONNEES_LOCAL = DOSSIER_LOCAL / "communes_coordonnees.parquet"
FICHIER_FINAL_LOCAL = DOSSIER_LOCAL / "voisinage_routier_20min.parquet"
FICHIER_ERREURS_LOCAL = DOSSIER_LOCAL / "erreurs_osrm.csv"


S3_COORDONNEES = (
    "s3://mangane/Projet_PDS_2026/processed/"
    "communes_coordonnees.parquet"
)
S3_FINAL = (
    "s3://mangane/Projet_PDS_2026/processed/"
    "voisinage_routier_20min.parquet"
)


MODE_TEST = False
NB_COMMUNES_TEST = 50


TAILLE_BATCH_ORIGINES = 250

UPLOAD_S3 = True


In [4]:

communes_insee = pd.read_csv(
    URL_INSEE,
    dtype={"CODGEO": str},
    low_memory=False
)

communes_insee = (
    communes_insee[["CODGEO", "LIBGEO"]]
    .drop_duplicates("CODGEO")
    .copy()
)

print("Nombre de communes INSEE :", len(communes_insee))
print("Doublons CODGEO :", communes_insee["CODGEO"].duplicated().sum())

communes_insee.head()


Nombre de communes INSEE : 34858
Doublons CODGEO : 0


,CODGEO,LIBGEO
0,01001,L'Abergement-Clémenciat
1,01002,L'Abergement-de-Varey
2,01004,Ambérieu-en-Bugey
3,01005,Ambérieux-en-Dombes
4,01006,Ambléon


On utilise l'API `geo.api.gouv.fr`.

Le point `mairie` est utilisé en priorité comme approximation du chef-lieu de la commune.  
Si ce point n'est pas disponible, on utilise le `centre` fourni par l'API.


In [8]:

def code_departement(codgeo):
    codgeo = str(codgeo)

    if codgeo.startswith(("2A", "2B")):
        return codgeo[:2]

    if codgeo.startswith("97"):
        return codgeo[:3]

    return codgeo[:2]


departements = sorted(
    communes_insee["CODGEO"]
    .dropna()
    .astype(str)
    .map(code_departement)
    .unique()
)

lignes = []

for dep in tqdm(departements):

    url = f"https://geo.api.gouv.fr/departements/{dep}/communes"

    params = {
        "fields": "code,nom,centre"
    }

    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()

    for commune in r.json():

        centre = commune.get("centre")

        if centre is None:
            continue

        lon, lat = centre["coordinates"]

        lignes.append({
            "CODGEO": str(commune["code"]),
            "LIBGEO_API": commune["nom"],
            "LONGITUDE": lon,
            "LATITUDE": lat
        })

coordonnees = pd.DataFrame(lignes)

print("Coordonnées récupérées :", len(coordonnees))
coordonnees.head()

100%|██████████| 100/100 [01:27<00:00,  1.14it/s]


Coordonnées récupérées : 34858


,CODGEO,LIBGEO_API,LONGITUDE,LATITUDE
0,01001,L'Abergement-Clémenciat,4.9306,46.1517
1,01002,L'Abergement-de-Varey,5.4247,46.0071
2,01004,Ambérieu-en-Bugey,5.3706,45.9575
3,01005,Ambérieux-en-Dombes,4.9119,45.9992
4,01006,Ambléon,5.5928,45.7483


In [10]:
# Fusion des coordonnées récupérées avec le référentiel INSEE

coordonnees = (
    coordonnees
    .drop_duplicates("CODGEO", keep="first")
)

communes = communes_insee.merge(
    coordonnees,
    on="CODGEO",
    how="left",
    validate="one_to_one"
)

print("Communes totales :", len(communes))

print(
    "Communes sans coordonnées :",
    communes["LONGITUDE"].isna().sum()
)

communes[
    communes["LONGITUDE"].isna()
].head(20)

Communes totales : 34858
Communes sans coordonnées : 0


,CODGEO,LIBGEO,LIBGEO_API,LONGITUDE,LATITUDE


In [11]:

communes_coord = (
    communes
    .dropna(subset=["LONGITUDE", "LATITUDE"])
    .copy()
)

communes_coord.to_parquet(
    FICHIER_COORDONNEES_LOCAL,
    index=False
)

print("Référentiel enregistré :", FICHIER_COORDONNEES_LOCAL)
print("Nombre de communes géolocalisées :", len(communes_coord))


Référentiel enregistré : data/processed/distances/communes_coordonnees.parquet
Nombre de communes géolocalisées : 34858



Construction de l'index spatial

On utilise un `BallTree` avec la distance de Haversine.

Cette étape sert uniquement à éviter des appels routiers inutiles :  
pour chaque commune d'origine, on sélectionne les communes situées à moins de 50 km à vol d'oiseau.

Le temps et la distance utilisés dans l'indicateur seront ensuite ceux calculés par OSRM.


In [12]:

RAYON_TERRE_KM = 6371.0088

# BallTree attend latitude puis longitude, exprimées en radians.
coord_rad = np.radians(
    communes_coord[["LATITUDE", "LONGITUDE"]]
    .to_numpy()
)

tree = BallTree(
    coord_rad,
    metric="haversine"
)

rayon_rad = RAYON_PREFILTRE_KM / RAYON_TERRE_KM

print("Index spatial construit.")


Index spatial construit.


Fonctions OSRM

In [ ]:

session_osrm = requests.Session()

def chunks(liste, taille):
    for i in range(0, len(liste), taille):
        yield liste[i:i + taille]


def poids_apl(temps_min):
    if pd.isna(temps_min):
        return np.nan
    if temps_min <= 10:
        return 1.0
    if temps_min <= 15:
        return 2 / 3
    if temps_min <= 20:
        return 1 / 3
    return 0.0


def classe_temps(temps_min):
    if pd.isna(temps_min):
        return None
    if temps_min <= 10:
        return "0-10 min"
    if temps_min <= 15:
        return "10-15 min"
    if temps_min <= 20:
        return "15-20 min"
    return ">20 min"


def requete_osrm_table(
    lon_origine,
    lat_origine,
    destinations
):
    """
    destinations : liste de tuples (lon, lat)

    Renvoie deux listes :
    - durées en secondes
    - distances en mètres
    """

    coords = [
        f"{lon_origine:.6f},{lat_origine:.6f}"
    ]

    coords += [
        f"{lon:.6f},{lat:.6f}"
        for lon, lat in destinations
    ]

    coord_string = ";".join(coords)

    destination_ids = ";".join(
        str(i)
        for i in range(1, len(coords))
    )

    url = (
        f"{OSRM_SERVER}/table/v1/"
        f"{OSRM_PROFILE}/{coord_string}"
    )

    params = {
        "sources": "0",
        "destinations": destination_ids,
        "annotations": "duration,distance",
        "exclude": "ferry",
    }

    derniere_erreur = None

    for tentative in range(1, MAX_RETRIES + 1):
        try:
            reponse = session_osrm.get(
                url,
                params=params,
                timeout=120
            )


            if reponse.status_code in {
                429, 500, 502, 503, 504
            }:
                raise RuntimeError(
                    f"HTTP {reponse.status_code}"
                )

            reponse.raise_for_status()
            data = reponse.json()

            if data.get("code") != "Ok":
                raise RuntimeError(
                    f"OSRM : {data.get('code')} "
                    f"- {data.get('message')}"
                )

            durees = data["durations"][0]
            distances = data["distances"][0]

            time.sleep(PAUSE_ENTRE_REQUETES)

            return durees, distances

        except Exception as e:
            derniere_erreur = str(e)

            attente = min(2 ** tentative, 30)
            time.sleep(attente)

    raise RuntimeError(
        f"Échec OSRM après {MAX_RETRIES} tentatives : "
        f"{derniere_erreur}"
    )


Test du serveur avant le calcul complet

In [14]:

# Petit test reproductible sur deux communes disponibles dans notre table.

test_points = communes_coord.head(2)

origine_test = test_points.iloc[0]
destination_test = test_points.iloc[1]

durees_test, distances_test = requete_osrm_table(
    origine_test["LONGITUDE"],
    origine_test["LATITUDE"],
    [(
        destination_test["LONGITUDE"],
        destination_test["LATITUDE"],
    )]
)

print("Origine :", origine_test["CODGEO"], origine_test["LIBGEO"])
print(
    "Destination :",
    destination_test["CODGEO"],
    destination_test["LIBGEO"],
)
print("Temps routier (min) :", durees_test[0] / 60)
print("Distance routière (km) :", distances_test[0] / 1000)


Origine : 01001 L'Abergement-Clémenciat
Destination : 01002 L'Abergement-de-Varey
Temps routier (min) : 54.99333333333333
Distance routière (km) : 51.069



Calcul des couples accessibles

Pour chaque commune :

- on cherche les communes candidates à moins de 50 km à vol d'oiseau ;
- le trajet vers la commune elle-même est fixé à 0 minute et 0 km ;
- les autres trajets sont envoyés à Metric-OSRM par petits groupes ;
- seuls les trajets routiers de 20 minutes maximum sont conservés.

Le traitement est sauvegardé par lots afin de pouvoir être repris après une interruption.


In [15]:

def calculer_origine(i):
    origine = communes_coord.iloc[i]

    # Recherche des voisines géographiques.
    indices, distances_rad = tree.query_radius(
        coord_rad[[i]],
        r=rayon_rad,
        return_distance=True,
        sort_results=True
    )

    indices = indices[0]
    distances_km = (
        distances_rad[0] * RAYON_TERRE_KM
    )

    # Dictionnaire index -> distance vol d'oiseau.
    dist_vo = {
        int(idx): float(dist)
        for idx, dist in zip(indices, distances_km)
    }

    # La commune elle-même est toujours accessible.
    lignes_origine = [{
        "CODGEO_origine": origine["CODGEO"],
        "LIBGEO_origine": origine["LIBGEO"],
        "CODGEO_destination": origine["CODGEO"],
        "LIBGEO_destination": origine["LIBGEO"],
        "distance_vol_oiseau_km": 0.0,
        "distance_route_km": 0.0,
        "temps_route_min": 0.0,
        "classe_temps": "0-10 min",
        "poids_distance": 1.0,
    }]

    # Autres communes candidates.
    candidats = [
        int(idx)
        for idx in indices
        if int(idx) != i
    ]

    for paquet_indices in chunks(
        candidats,
        MAX_DESTINATIONS_PAR_REQUETE
    ):
        dest = communes_coord.iloc[paquet_indices]

        destinations = list(
            zip(
                dest["LONGITUDE"].astype(float),
                dest["LATITUDE"].astype(float),
            )
        )

        durees_sec, distances_m = requete_osrm_table(
            float(origine["LONGITUDE"]),
            float(origine["LATITUDE"]),
            destinations,
        )

        for idx_dest, duree_sec, distance_m in zip(
            paquet_indices,
            durees_sec,
            distances_m,
        ):
            # None = aucune liaison routière trouvée.
            if duree_sec is None or distance_m is None:
                continue

            temps_min = float(duree_sec) / 60.0

            # Seul le voisinage utile à l'APL est conservé.
            if temps_min > 20:
                continue

            destination = communes_coord.iloc[idx_dest]

            lignes_origine.append({
                "CODGEO_origine": origine["CODGEO"],
                "LIBGEO_origine": origine["LIBGEO"],
                "CODGEO_destination": destination["CODGEO"],
                "LIBGEO_destination": destination["LIBGEO"],
                "distance_vol_oiseau_km": dist_vo[idx_dest],
                "distance_route_km": float(distance_m) / 1000.0,
                "temps_route_min": temps_min,
                "classe_temps": classe_temps(temps_min),
                "poids_distance": poids_apl(temps_min),
            })

    return lignes_origine


Reprise automatique des calculs déjà effectués

In [16]:

fichiers_existants = sorted(
    DOSSIER_BATCHES.glob("part_*.parquet")
)

origines_deja_traitees = set()

for fichier in fichiers_existants:
    tmp = pd.read_parquet(
        fichier,
        columns=["CODGEO_origine"]
    )
    origines_deja_traitees.update(
        tmp["CODGEO_origine"].astype(str).unique()
    )

print(
    "Origines déjà sauvegardées :",
    len(origines_deja_traitees)
)

if fichiers_existants:
    numeros = [
        int(f.stem.split("_")[1])
        for f in fichiers_existants
    ]
    prochain_numero_batch = max(numeros) + 1
else:
    prochain_numero_batch = 0


Origines déjà sauvegardées : 0


Lancement du calcul

In [17]:

indices_a_traiter = [
    i
    for i, code in enumerate(communes_coord["CODGEO"])
    if code not in origines_deja_traitees
]

if MODE_TEST:
    indices_a_traiter = indices_a_traiter[:NB_COMMUNES_TEST]

print("Origines à traiter :", len(indices_a_traiter))


Origines à traiter : 34858



Assemblage de la table finale

Tous les fichiers intermédiaires sont réunis.

On supprime d'éventuels doublons, puis on vérifie qu'il existe au plus une ligne par couple origine-destination.


In [19]:

fichiers_batches = sorted(
    DOSSIER_BATCHES.glob("part_*.parquet")
)

if not fichiers_batches:
    raise RuntimeError(
        "Aucun batch n'a encore été produit."
    )

voisinage = pd.concat(
    [
        pd.read_parquet(f)
        for f in fichiers_batches
    ],
    ignore_index=True
)

voisinage = (
    voisinage
    .drop_duplicates(
        ["CODGEO_origine", "CODGEO_destination"]
    )
    .sort_values(
        ["CODGEO_origine", "temps_route_min"]
    )
    .reset_index(drop=True)
)

print("Nombre de couples :", len(voisinage))
print(
    "Origines couvertes :",
    voisinage["CODGEO_origine"].nunique()
)
print(
    "Destinations distinctes :",
    voisinage["CODGEO_destination"].nunique()
)
print(
    "Doublons origine-destination :",
    voisinage.duplicated(
        ["CODGEO_origine", "CODGEO_destination"]
    ).sum()
)

voisinage.head()


Nombre de couples : 409737
Origines couvertes : 15500
Destinations distinctes : 19086
Doublons origine-destination : 0


,CODGEO_origine,LIBGEO_origine,CODGEO_destination,LIBGEO_destination,distance_vol_oiseau_km,distance_route_km,temps_route_min,classe_temps,poids_distance
0,01001,L'Abergement-Clémenciat,01001,L'Abergement-Clémenciat,0.000000,0.0000,0.000000,0-10 min,1.0
1,01001,L'Abergement-Clémenciat,01146,Dompierre-sur-Chalaronne,2.564389,5.2653,7.496667,0-10 min,1.0
2,01001,L'Abergement-Clémenciat,01412,Sulignat,3.257992,5.7110,8.463333,0-10 min,1.0
3,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,3.379304,6.8761,9.068333,0-10 min,1.0
4,01001,L'Abergement-Clémenciat,01188,Illiat,4.827127,7.2143,9.826667,0-10 min,1.0


Contrôles

In [20]:

print("Temps routier maximal :")
print(voisinage["temps_route_min"].max())

print("\nDistribution des classes de temps :")
print(
    voisinage["classe_temps"]
    .value_counts(dropna=False)
)

print("\nDistribution des poids :")
print(
    voisinage["poids_distance"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNombre de destinations accessibles par origine :")
print(
    voisinage
    .groupby("CODGEO_origine")
    .size()
    .describe()
)


Temps routier maximal :
20.0

Distribution des classes de temps :
classe_temps
15-20 min    203650
10-15 min    125696
0-10 min      80391
Name: count, dtype: int64

Distribution des poids :
poids_distance
0.333333    203650
0.666667    125696
1.000000     80391
Name: count, dtype: int64

Nombre de destinations accessibles par origine :
count    15500.000000
mean        26.434645
std         16.143041
min          1.000000
25%         14.000000
50%         24.000000
75%         36.000000
max        100.000000
dtype: float64


In [21]:

# Vérifications essentielles.
assert (
    voisinage["temps_route_min"] <= 20 + 1e-9
).all()

assert not voisinage.duplicated(
    ["CODGEO_origine", "CODGEO_destination"]
).any()

assert set(
    voisinage["poids_distance"].dropna().unique()
).issubset({
    1.0,
    2 / 3,
    1 / 3,
})

print("Contrôles validés.")


Contrôles validés.


Sauvegarde finale

In [22]:

voisinage.to_parquet(
    FICHIER_FINAL_LOCAL,
    index=False
)

print(
    "Table locale enregistrée :",
    FICHIER_FINAL_LOCAL
)

if UPLOAD_S3:
    try:
        communes_coord.to_parquet(
            S3_COORDONNEES,
            index=False
        )

        voisinage.to_parquet(
            S3_FINAL,
            index=False
        )

        print("Fichiers envoyés sur MinIO :")
        print("-", S3_COORDONNEES)
        print("-", S3_FINAL)

    except Exception as e:
        print(
            "L'envoi MinIO a échoué, "
            "mais les fichiers locaux sont conservés."
        )
        print("Erreur :", e)


Table locale enregistrée : data/processed/distances/voisinage_routier_20min.parquet
Fichiers envoyés sur MinIO :
- s3://mangane/Projet_PDS_2026/processed/communes_coordonnees.parquet
- s3://mangane/Projet_PDS_2026/processed/voisinage_routier_20min.parquet



## Remarques méthodologiques

- On reste sur une logique d'accès strictement routier
- Le rayon de 50 km à vol d'oiseau est un préfiltre informatique et ne remplace pas le temps routier
- Les fichiers intermédiaires permettent de reprendre le calcul sans recommencer les communes déjà traitées
